In [8]:
import sqlite3

conn = sqlite3.connect('data/stories.db')
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

bookId = 19
fixed_query = """
SELECT b.*,
  (SELECT GROUP_CONCAT(c.name) FROM book_categories bc JOIN categories c ON bc.category_id = c.id WHERE bc.book_id = b.id) as category_names,
  (SELECT GROUP_CONCAT(bc.category_id) FROM book_categories bc WHERE bc.book_id = b.id) as category_ids,
  (SELECT GROUP_CONCAT(t.name) FROM book_tags bt JOIN tags t ON bt.tag_id = t.id WHERE bt.book_id = b.id) as tag_names
FROM books b
WHERE b.id = ?
"""

try:
    cursor.execute(fixed_query, (bookId,))
    row = cursor.fetchone()
    if row:
        print("Success! Row retrieved:")
        print(dict(row))
    else:
        print("Book not found for ID:", bookId)
except Exception as e:
    print("SQL Error:", e)

conn.close()

Success! Row retrieved:
{'id': 19, 'title': 'Soul Hunter', 'author': 'Aaron Dembski Bowden', 'description': 'The Night Lords are one of the most feared legions of Chaos Space Marines. Remorseless hunters and killers, they relentlessly battle the Imperium of Man to avenge the death of their Primarch Konrad Curze. Their dark crusade takes them to the valuable world of Crythe Primus, where they will fight Imperial forces to claim the planet. But will the allegiance with their cohorts in the Black Legion last long enough for them to be victorious?', 'publisher': None, 'language': 'en', 'isbn': None, 'published_date': None, 'page_count': None, 'est_read_minutes': None, 'cover_image_url': '/uploads/5fd50aa0-c216-433f-b7fa-beab29f177de.jpg', 'file_url': '/uploads/5fd50aa0-c216-433f-b7fa-beab29f177de.epub', 'file_type': 'epub', 'status': 'published', 'visibility': 'public', 'uploaded_by': None, 'approved_by': None, 'created_at': '2026-07-20 17:45:40', 'updated_at': '2026-07-20 17:45:40', 'chan

In [5]:
import sqlite3
import os

DB_PATH = 'data/stories.db'
OUTPUT = 'scratch/remote_books.sql'

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# Get all books
cursor.execute("""
    SELECT b.id, b.title, b.author, b.description, b.file_url, b.cover_image_url,
           b.file_type, b.status, b.visibility, b.channel_type, b.is_user_submission, b.submission_status,
           c.slug as cat_slug
    FROM books b
    LEFT JOIN book_categories bc ON bc.book_id = b.id
    LEFT JOIN categories c ON c.id = bc.category_id
    ORDER BY b.id
""")
rows = cursor.fetchall()
conn.close()

lines = []
lines.append('-- SQL Script to insert uploaded books into remote Cloudflare D1')
lines.append('PRAGMA foreign_keys = OFF;')
lines.append('')

seen_books = set()
for row in rows:
    bid = row['id']
    
    # Sanitize text fields for SQL
    def esc(v):
        if v is None: return 'NULL'
        return "'" + str(v).replace("'", "''") + "'"
    
    if bid not in seen_books:
        seen_books.add(bid)
        lines.append(
            f"INSERT OR IGNORE INTO books (id, title, author, description, file_url, cover_image_url, file_type, status, visibility, channel_type, is_user_submission, submission_status) "
            f"VALUES ({bid}, {esc(row['title'])}, {esc(row['author'])}, {esc(row['description'])}, "
            f"{esc(row['file_url'])}, {esc(row['cover_image_url'])}, {esc(row['file_type'])}, "
            f"'published', 'public', {esc(row['channel_type'])}, 0, 'approved');"
        )
    
    # Use slug-based subquery so category_id matches the remote DB's auto-generated IDs
    cat_slug = row['cat_slug']
    if cat_slug:
        lines.append(
            f"INSERT OR IGNORE INTO book_categories (book_id, category_id) "
            f"SELECT {bid}, id FROM categories WHERE slug = {esc(cat_slug)} LIMIT 1;"
        )

lines.append('')
lines.append('PRAGMA foreign_keys = ON;')

with open(OUTPUT, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f'✅ Generated {OUTPUT} with {len(seen_books)} books using slug-based category lookups.')
print(f'   Total lines: {len(lines)}')

✅ Generated scratch/remote_books.sql with 430 books using slug-based category lookups.
   Total lines: 865


In [ ]:
# [Notebook MCP] Internal check: Waking up kernel connection or awaiting UI Kernel Selection...